In [ ]:
import os
import gymnasium as gym
import matplotlib.pyplot as plt
import numpy as np
from stable_baselines3 import PPO
from stable_baselines3.common.evaluation import evaluate_policy
from stable_baselines3.common.vec_env import DummyVecEnv

save_path = "outputs/checkpoints"
os.makedirs(save_path, exist_ok=True)

target_pi2_reward = 250
target_pi1_reward = 490
seed = 0
pi2_steps = 14336

checkpoint_interval = 1024

def make_env(env_id: str, seed: int) -> gym.Env:
    env = gym.make(env_id)
    env.reset(seed=seed)
    return env

print(f"\n=== CartPole seed {seed} ===")

rng = np.random.default_rng(seed)
env_train = DummyVecEnv([lambda: make_env("CartPole-v1", seed=rng.integers(0, 1000000))])
n_eval_episodes = 20
env_eval  = DummyVecEnv([lambda: make_env("CartPole-v1", seed=rng.integers(0, 1000000)) for _ in range(n_eval_episodes)])

steps_log, mean_log, std_log = [], [], []

model = PPO(
    "MlpPolicy",
    env_train,
    verbose=0,
    device="cpu",
    seed=seed,
    learning_rate=1e-4,
    n_steps=256,
    batch_size=64,
    n_epochs=5,
    ent_coef=0.01,
    gamma=0.99,
)

total_steps = 0
pi2_saved = False
pi1_saved = False

for _ in range(0, 100_000, checkpoint_interval):
    model.learn(
        total_timesteps=checkpoint_interval,
        reset_num_timesteps=False
    )
    total_steps += checkpoint_interval
    mean_reward, std_reward = evaluate_policy(
        model, env_eval, n_eval_episodes=n_eval_episodes, deterministic=True
    )
    steps_log.append(total_steps)
    mean_log.append(mean_reward)
    std_log.append(std_reward)
    print(f"{total_steps} steps: {mean_reward:.1f} +/- {std_reward:.1f}")

    # save pi2 at target steps
    if total_steps == pi2_steps and not pi2_saved:
        model.save(os.path.join(save_path, f"pi2_cartpole_seed{seed}"))
        print(f">>> pi2 saved at {total_steps} steps: {mean_reward:.1f} +/- {std_reward:.1f}")
        pi2_saved = True

    # save pi1 once policy is excellent
    if mean_reward >= target_pi1_reward and not pi1_saved:
        model.save(os.path.join(save_path, f"pi1_cartpole_seed{seed}"))
        print(f">>> pi1 saved at {total_steps} steps: {mean_reward:.1f} +/- {std_reward:.1f}")
        pi1_saved = True

    if pi1_saved:
        print(f"Both policies saved for seed {seed}, stopping")
        break

# pick the best checkpoint for pi2 on the next run
if steps_log:
    mean_array = np.asarray(mean_log)
    best_pi2_idx = int(np.argmin(np.abs(mean_array - target_pi2_reward)))
    best_pi2_steps = steps_log[best_pi2_idx]
    best_pi2_reward = mean_log[best_pi2_idx]
    print(
        f"Suggested pi2_steps for the next run: {best_pi2_steps} "
        f"(mean reward {best_pi2_reward:.1f}, target {target_pi2_reward})"
    )
    pi2_steps = best_pi2_steps

# plot
plt.figure(figsize=(10, 5))
plt.plot(steps_log, mean_log, label="mean reward")
plt.fill_between(
    steps_log,
    [m - s for m, s in zip(mean_log, std_log)],
    [m + s for m, s in zip(mean_log, std_log)],
    alpha=0.3, label="std"
)
plt.axhline(target_pi2_reward, color="red",   linestyle="--", label=f"pi2 target ~{target_pi2_reward}")
plt.axhline(target_pi1_reward, color="green", linestyle="--", label=f"pi1 target ~{target_pi1_reward}")
plt.xlabel("steps")
plt.ylabel("reward")
plt.title(f"CartPole seed {seed} training curve")
plt.legend()
plt.show()

print("\nDone! Saved policies:")
print(f"  pi1_cartpole_seed{seed}.zip  |  pi2_cartpole_seed{seed}.zip")

In [ ]:
target_pi2_reward = -900
target_pi1_reward = -200
checkpoint_interval = 1024

seed = 0
pi2_steps = 160768

print(f"\n=== Pendulum seed {seed} ===")

rng = np.random.default_rng(seed)
env_train = DummyVecEnv([lambda: make_env("Pendulum-v1", seed=rng.integers(0, 1000000))])
n_eval_episodes = 20
env_eval  = DummyVecEnv([lambda: make_env("Pendulum-v1", seed=rng.integers(0, 1000000)) for _ in range(n_eval_episodes)])

steps_log, mean_log, std_log = [], [], []

model = PPO(
    "MlpPolicy", env_train, verbose=0, device="cpu", seed=seed,
    learning_rate=3e-4, n_steps=2048, batch_size=64,
    n_epochs=5, ent_coef=0.01, gamma=0.99,
)

total_steps = 0
pi2_saved = False
pi1_saved = False

for _ in range(0, 1_000_000, checkpoint_interval):
    model.learn(
        total_timesteps=checkpoint_interval,
        reset_num_timesteps=False
    )
    total_steps += checkpoint_interval
    mean_reward, std_reward = evaluate_policy(
        model, env_eval, n_eval_episodes=n_eval_episodes, deterministic=True
    )
    steps_log.append(total_steps)
    mean_log.append(mean_reward)
    std_log.append(std_reward)
    print(f"{total_steps} steps: {mean_reward:.1f} +/- {std_reward:.1f}")

    # save pi2 at target steps
    if total_steps == pi2_steps and not pi2_saved:
        model.save(os.path.join(save_path, f"pi2_pendulum_seed{seed}"))
        print(f">>> pi2 saved at {total_steps} steps: {mean_reward:.1f} +/- {std_reward:.1f}")
        pi2_saved = True

    # save pi1 once policy is excellent
    if mean_reward >= target_pi1_reward and not pi1_saved:
        model.save(os.path.join(save_path, f"pi1_pendulum_seed{seed}"))
        print(f">>> pi1 saved at {total_steps} steps: {mean_reward:.1f} +/- {std_reward:.1f}")
        pi1_saved = True

    if pi1_saved:
        print(f"Both policies saved for seed {seed}, stopping")
        break

# pick the best checkpoint for pi2 on the next run
if steps_log:
    mean_array = np.asarray(mean_log)
    best_pi2_idx = int(np.argmin(np.abs(mean_array - target_pi2_reward)))
    best_pi2_steps = steps_log[best_pi2_idx]
    best_pi2_reward = mean_log[best_pi2_idx]
    print(
        f"Suggested pi2_steps for the next run: {best_pi2_steps} "
        f"(mean reward {best_pi2_reward:.1f}, target {target_pi2_reward})"
    )
    pi2_steps = best_pi2_steps

# plot
plt.figure(figsize=(10, 5))
plt.plot(steps_log, mean_log, label="mean reward")
plt.fill_between(
    steps_log,
    [m - s for m, s in zip(mean_log, std_log)],
    [m + s for m, s in zip(mean_log, std_log)],
    alpha=0.3, label="std"
)
plt.axhline(target_pi2_reward, color="red",   linestyle="--", label=f"pi2 target ~{target_pi2_reward}")
plt.axhline(target_pi1_reward, color="green", linestyle="--", label=f"pi1 target ~{target_pi1_reward}")
plt.xlabel("steps")
plt.ylabel("reward")
plt.title(f"Pendulum seed {seed} training curve")
plt.legend()
plt.show()

print("\nDone! Saved policies:")
print(f"  pi1_pendulum_seed{seed}.zip  |  pi2_pendulum_seed{seed}.zip")